# Applied CNNs: Parcel Label Digit Recognition

Build a complete image-classification prototype with PyTorch: inspect the data, track tensor shapes, learn convolutional filters, validate the model, analyze errors, add a confidence-based review policy, save a reproducible checkpoint, and measure local inference latency.

**Project story:** A warehouse camera captures one handwritten routing digit (`0` to `9`) from a parcel label. The model proposes a conveyor lane. Low-confidence cases are sent for manual review.

This notebook uses scikit-learn's small Digits dataset so it runs quickly on CPU and does not need an internet download. It demonstrates an engineering workflow; it is not evidence that the model is ready for a real warehouse.


### Workflow

`problem contract → data checks → convolution → CNN → validation → test → error analysis → decision policy → checkpoint → latency`

## 1. Environment and reproducibility

We import the required libraries, fix random seeds, and select an available device. Reproducibility can still vary across hardware and library versions, so production experiments should also record the package environment and code revision.

In [ ]:
# WHAT: Import libraries, record versions, fix seeds, and select a device.
# WHY: A reproducible experiment starts with an explicit environment and random state.
# OUTPUT: Package versions and the device used by PyTorch.

from copy import deepcopy
from pathlib import Path
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_digits
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"PyTorch: {torch.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"Device: {device}")

## 2. Define the ML contract

- **Input:** one grayscale image with shape `[1, 8, 8]` and pixel values scaled to `[0, 1]`.
- **Target:** one integer class index from `0` to `9`.
- **Model output:** ten raw logits, one for each class.
- **Offline checks:** accuracy, per-class precision/recall, confusion matrix, and error inspection.
- **Decision policy:** accept a prediction only when confidence is high enough; otherwise request review.

The confidence threshold is a product decision that should be selected on validation data using the cost of wrong automation and manual review. We will use several values only to study the trade-off.

## 3. Load and inspect the images

The Digits dataset contains `8 × 8` grayscale images. Pixel values range from `0` to `16`. We inspect shape, range, target classes, and class balance before building a model.

In [ ]:
# WHAT: Load the digit images and inspect the raw data contract.
# WHY: Shape, range, dtype, and class balance determine preprocessing and model design.
# OUTPUT: Dataset shapes, pixel statistics, class names, and class counts.

digits = load_digits()
X_raw = digits.images
y_raw = digits.target.astype(np.int64)
class_names = np.array([str(value) for value in digits.target_names])

unique_classes, class_counts = np.unique(y_raw, return_counts=True)

print(f"Images: {X_raw.shape} | dtype={X_raw.dtype}")
print(f"Targets: {y_raw.shape} | dtype={y_raw.dtype}")
print(f"Pixel range: [{X_raw.min():.1f}, {X_raw.max():.1f}]")
print(f"Classes: {class_names.tolist()}")
print("Class counts:", dict(zip(unique_classes.tolist(), class_counts.tolist())))

assert X_raw.ndim == 3 and X_raw.shape[1:] == (8, 8)
assert set(unique_classes.tolist()) == set(range(10))

In [ ]:
# WHAT: Display one example from each class.
# WHY: Visual inspection can reveal label problems, orientation issues, or unexpected artifacts.
# OUTPUT: A row of ten labeled digit images.

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for digit, axis in enumerate(axes.flat):
    sample_index = np.flatnonzero(y_raw == digit)[0]
    axis.imshow(X_raw[sample_index], cmap="gray_r", vmin=0, vmax=16)
    axis.set_title(f"Label: {digit}")
    axis.axis("off")

plt.suptitle("One raw example from each class")
plt.tight_layout()
plt.show()

## 4. Create train, validation, and test splits

The split is stratified so each class keeps a similar proportion. The training set updates weights, validation selects the best epoch and engineering choices, and the test set stays untouched until final evaluation.

We divide by the known maximum pixel value `16`, convert to `float32`, and add the grayscale channel dimension. The final image contract is `[N, 1, 8, 8]`.

In [ ]:
# WHAT: Split the data, scale pixels, add the channel dimension, and create tensors.
# WHY: Correct isolation and a stable NCHW contract prevent leakage and shape bugs.
# OUTPUT: Train, validation, and test tensors with verified shapes and ranges.

X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
    X_raw,
    y_raw,
    test_size=0.30,
    stratify=y_raw,
    random_state=SEED,
)
X_valid_raw, X_test_raw, y_valid, y_test = train_test_split(
    X_temp_raw,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=SEED,
)

def prepare_images(images):
    scaled = images.astype(np.float32) / 16.0
    return torch.from_numpy(scaled).unsqueeze(1)

X_train = prepare_images(X_train_raw)
X_valid = prepare_images(X_valid_raw)
X_test = prepare_images(X_test_raw)

y_train_tensor = torch.from_numpy(y_train)
y_valid_tensor = torch.from_numpy(y_valid)
y_test_tensor = torch.from_numpy(y_test)

print("Train:", X_train.shape, y_train_tensor.shape)
print("Valid:", X_valid.shape, y_valid_tensor.shape)
print("Test: ", X_test.shape, y_test_tensor.shape)
print(f"Scaled train range: [{X_train.min():.2f}, {X_train.max():.2f}]")

assert X_train.shape[1:] == (1, 8, 8)
assert X_train.dtype == torch.float32
assert y_train_tensor.dtype == torch.int64
assert X_train.min() >= 0 and X_train.max() <= 1

## 5. Build mini-batches

Only the training loader shuffles. Validation and test order do not affect metrics. `num_workers=0` keeps this notebook portable across operating systems and hosted notebook environments.

In [ ]:
# WHAT: Wrap tensors in datasets and create deterministic data loaders.
# WHY: Mini-batches make training efficient while preserving a clear data boundary.
# OUTPUT: One inspected batch with image and label shapes.

BATCH_SIZE = 64
loader_generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    TensorDataset(X_train, y_train_tensor),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    generator=loader_generator,
)
valid_loader = DataLoader(
    TensorDataset(X_valid, y_valid_tensor),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)
test_loader = DataLoader(
    TensorDataset(X_test, y_test_tensor),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

batch_images, batch_labels = next(iter(train_loader))
print("Image batch:", batch_images.shape)
print("Label batch:", batch_labels.shape)
print("First labels:", batch_labels[:10].tolist())

assert batch_images.ndim == 4
assert batch_images.shape[1:] == (1, 8, 8)

## 6. See one filter work

A convolution filter multiplies a local image patch by small weights and sums the products. The same weights slide across all spatial positions. This fixed kernel highlights vertical intensity changes only to build intuition; the CNN will learn its own filters through backpropagation.

PyTorch `conv2d` uses cross-correlation: it slides the stored kernel without flipping it. Deep learning libraries still use the conventional name “convolution.”

In [ ]:
# WHAT: Apply a fixed vertical-edge kernel to one normalized image.
# WHY: A visible feature map makes local filtering and weight sharing concrete.
# OUTPUT: The original digit beside the filter response.

example_image = X_train[0:1]  # [N=1, C=1, H=8, W=8]
vertical_edge_kernel = torch.tensor(
    [[-1.0, 0.0, 1.0],
     [-1.0, 0.0, 1.0],
     [-1.0, 0.0, 1.0]],
    dtype=torch.float32,
).reshape(1, 1, 3, 3)

edge_response = F.conv2d(example_image, vertical_edge_kernel, padding=1)

fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(example_image[0, 0], cmap="gray_r")
axes[0].set_title(f"Input digit: {y_train[0]}")
axes[1].imshow(edge_response[0, 0], cmap="coolwarm")
axes[1].set_title("Vertical-edge response")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()

print("Input shape:", tuple(example_image.shape))
print("Kernel shape:", tuple(vertical_edge_kernel.shape))
print("Feature-map shape:", tuple(edge_response.shape))

## 7. Calculate convolution output shapes

For one spatial dimension with dilation `D`:

\[
\text{output} = \left\lfloor
\frac{\text{input} + 2P - D(K-1) - 1}{S} + 1
\right\rfloor
\]

Here `K` is kernel size, `P` is padding, and `S` is stride. With `K=3`, `P=1`, and `S=1`, an `8 × 8` feature map stays `8 × 8`. A `2 × 2` max-pooling layer with stride 2 then halves it to `4 × 4`.

In [ ]:
# WHAT: Implement the full convolution-size formula and verify common cases.
# WHY: Shape calculations prevent incorrect flatten dimensions and wasted debugging time.
# OUTPUT: Verified output sizes for same-size and downsampling convolutions.

def conv_output_size(size, kernel_size, padding=0, stride=1, dilation=1):
    numerator = size + 2 * padding - dilation * (kernel_size - 1) - 1
    return numerator // stride + 1

same_size = conv_output_size(8, kernel_size=3, padding=1, stride=1)
downsampled = conv_output_size(8, kernel_size=3, padding=1, stride=2)

print("8x8 with K=3, P=1, S=1 ->", same_size, "x", same_size)
print("8x8 with K=3, P=1, S=2 ->", downsampled, "x", downsampled)

assert same_size == 8
assert downsampled == 4

## 8. Define a small CNN

The feature extractor increases channels while reducing spatial size:

```text
[N, 1, 8, 8]
  → Conv 1→16 + ReLU + Pool
[N, 16, 4, 4]
  → Conv 16→32 + ReLU + Pool
[N, 32, 2, 2]
  → Flatten
[N, 128]
  → Linear 128→64 → ReLU → Dropout → Linear 64→10
[N, 10 raw logits]
```

The model does **not** apply softmax in `forward`. `CrossEntropyLoss` expects raw logits and performs a numerically stable log-softmax internally. Softmax is used later for interpretation and the review policy.

In [ ]:
# WHAT: Define a compact CNN with separate feature and classifier blocks.
# WHY: A clear architecture and explicit config make shape checks and reloads easier.
# OUTPUT: A reusable PyTorch model class that returns ten raw logits.

class SmallCNN(nn.Module):
    def __init__(self, num_classes=10, dropout=0.20):
        super().__init__()
        self.num_classes = num_classes
        self.dropout_rate = dropout

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 2 * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes),
        )

    def forward(self, images):
        features = self.features(images)
        return self.classifier(features)

model = SmallCNN(num_classes=len(class_names), dropout=0.20).to(device)
print(model)

In [ ]:
# WHAT: Trace layer outputs, verify the final contract, and count trainable parameters.
# WHY: A dummy forward pass catches architecture errors before a training run begins.
# OUTPUT: A layer-by-layer shape trace and the total parameter count.

shape_trace = []
hooks = []

def capture_shape(name):
    def hook(_module, _inputs, output):
        shape_trace.append((name, tuple(output.shape)))
    return hook

for name, layer in model.named_modules():
    if isinstance(layer, (nn.Conv2d, nn.MaxPool2d, nn.Flatten, nn.Linear)):
        hooks.append(layer.register_forward_hook(capture_shape(name)))

with torch.inference_mode():
    dummy_logits = model(torch.zeros(4, 1, 8, 8, device=device))

for handle in hooks:
    handle.remove()

trainable_parameters = sum(
    parameter.numel() for parameter in model.parameters() if parameter.requires_grad
)

for layer_name, output_shape in shape_trace:
    print(f"{layer_name:18s} -> {output_shape}")
print(f"Trainable parameters: {trainable_parameters:,}")

assert dummy_logits.shape == (4, 10)

## 9. Train with validation-based model selection

The training mini-batch order is:

`zero gradients → forward → loss → backward → optimizer step`

Validation uses both `model.eval()` and `torch.inference_mode()`. Evaluation mode controls layers such as dropout; inference mode avoids building gradient graphs.

We use AdamW with `1e-3` as a starting learning rate, keep the state with the lowest validation loss, and stop when validation has not improved for several epochs. A learning rate is an experimental starting point, not a rule tied only to the loss function.

In [ ]:
# WHAT: Define reusable training and evaluation functions.
# WHY: Separating gradient updates from evaluation avoids mode and metric mistakes.
# OUTPUT: Functions that return sample-weighted loss, accuracy, labels, predictions, and logits.

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_samples += batch_size

    return total_loss / total_samples, total_correct / total_samples


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    all_labels = []
    all_predictions = []
    all_logits = []

    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            predictions = logits.argmax(dim=1)

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_correct += (predictions == labels).sum().item()
            total_samples += batch_size
            all_labels.append(labels.cpu())
            all_predictions.append(predictions.cpu())
            all_logits.append(logits.cpu())

    return {
        "loss": total_loss / total_samples,
        "accuracy": total_correct / total_samples,
        "labels": torch.cat(all_labels),
        "predictions": torch.cat(all_predictions),
        "logits": torch.cat(all_logits),
    }

In [ ]:
# WHAT: Train the CNN, select the lowest validation-loss state, and stop on a plateau.
# WHY: The last epoch is not automatically the best generalizing epoch.
# OUTPUT: Epoch metrics, the best epoch, and an in-memory copy of the best weights.

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

MAX_EPOCHS = 30
PATIENCE = 6
history = {"train_loss": [], "train_accuracy": [], "valid_loss": [], "valid_accuracy": []}
best_valid_loss = float("inf")
best_epoch = 0
best_state = None
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss, train_accuracy = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )
    valid_metrics = evaluate(model, valid_loader, criterion, device)

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["valid_loss"].append(valid_metrics["loss"])
    history["valid_accuracy"].append(valid_metrics["accuracy"])

    improved = valid_metrics["loss"] < best_valid_loss - 1e-4
    marker = " *" if improved else ""
    print(
        f"Epoch {epoch:02d} | "
        f"train loss {train_loss:.4f} acc {train_accuracy:.3f} | "
        f"valid loss {valid_metrics['loss']:.4f} acc {valid_metrics['accuracy']:.3f}{marker}"
    )

    if improved:
        best_valid_loss = valid_metrics["loss"]
        best_epoch = epoch
        best_state = deepcopy({key: value.cpu() for key, value in model.state_dict().items()})
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping after {epoch} epochs.")
        break

assert best_state is not None
print(f"Best epoch: {best_epoch} | validation loss: {best_valid_loss:.4f}")

In [ ]:
# WHAT: Plot training and validation learning curves.
# WHY: Curves reveal underfitting, overfitting, instability, and the effect of early stopping.
# OUTPUT: Side-by-side loss and accuracy curves with the selected epoch marked.

completed_epochs = np.arange(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(completed_epochs, history["train_loss"], label="train")
axes[0].plot(completed_epochs, history["valid_loss"], label="validation")
axes[0].axvline(best_epoch, color="black", linestyle="--", alpha=0.6, label="best epoch")
axes[0].set(title="Loss", xlabel="Epoch", ylabel="Cross-entropy")
axes[0].legend()

axes[1].plot(completed_epochs, history["train_accuracy"], label="train")
axes[1].plot(completed_epochs, history["valid_accuracy"], label="validation")
axes[1].axvline(best_epoch, color="black", linestyle="--", alpha=0.6, label="best epoch")
axes[1].set(title="Accuracy", xlabel="Epoch", ylabel="Accuracy", ylim=(0, 1.02))
axes[1].legend()

plt.tight_layout()
plt.show()

## 10. Restore the best state and evaluate once on test data

The test split has not influenced training or epoch selection. We now restore the best validation state and compute final test metrics.

Accuracy is only the first view. The classification report and confusion matrix show which classes are weak and which pairs are confused.

In [ ]:
# WHAT: Restore the best weights and calculate final test predictions.
# WHY: Final reporting should use the selected model, not whichever state happened to train last.
# OUTPUT: Test loss, test accuracy, and tensors used by later analysis.

model.load_state_dict(best_state)
model.to(device)
test_metrics = evaluate(model, test_loader, criterion, device)

y_test_eval = test_metrics["labels"].numpy()
test_predictions = test_metrics["predictions"].numpy()
test_probabilities = torch.softmax(test_metrics["logits"], dim=1).numpy()
test_confidences = test_probabilities.max(axis=1)

print(f"Test loss: {test_metrics['loss']:.4f}")
print(f"Test accuracy: {test_metrics['accuracy']:.3%}")

In [ ]:
# WHAT: Print per-class metrics and draw the confusion matrix.
# WHY: Aggregate accuracy can hide a class with poor precision or recall.
# OUTPUT: Precision, recall, F1-score, support, and a labeled confusion matrix.

print(
    classification_report(
        y_test_eval,
        test_predictions,
        labels=np.arange(10),
        target_names=class_names,
        digits=3,
        zero_division=0,
    )
)

matrix = confusion_matrix(y_test_eval, test_predictions, labels=np.arange(10))
plt.figure(figsize=(8, 6))
sns.heatmap(
    matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
)
plt.xlabel("Predicted digit")
plt.ylabel("True digit")
plt.title("Test confusion matrix")
plt.tight_layout()
plt.show()

## 11. Inspect mistakes instead of only tuning the model

Metrics summarize behavior; examples help explain it. For each mistake, inspect the image, true class, predicted class, and confidence. Look for ambiguous writing, weak strokes, crop problems, label noise, or a repeated confusion pattern.

A model change is only one possible response. The better fix may be improved labels, more representative data, a safer threshold, or a corrected preprocessing step.

In [ ]:
# WHAT: Display up to twelve misclassified test images with confidence.
# WHY: Error slices connect summary metrics to concrete data and failure modes.
# OUTPUT: An error gallery, or a message if no errors are present.

mistake_indices = np.flatnonzero(test_predictions != y_test_eval)
print(f"Misclassified examples: {len(mistake_indices)} / {len(y_test_eval)}")

if len(mistake_indices) == 0:
    print("No mistakes to display in this split.")
else:
    shown = mistake_indices[:12]
    fig, axes = plt.subplots(3, 4, figsize=(9, 7))
    for axis in axes.flat:
        axis.axis("off")
    for axis, index in zip(axes.flat, shown):
        axis.imshow(X_test_raw[index], cmap="gray_r", vmin=0, vmax=16)
        axis.set_title(
            f"true={y_test_eval[index]} pred={test_predictions[index]}\n"
            f"confidence={test_confidences[index]:.2f}"
        )
        axis.axis("off")
    plt.suptitle("Test errors to investigate")
    plt.tight_layout()
    plt.show()

## 12. Add a confidence-based review policy

Softmax converts logits into values that sum to one, but neural-network confidence is not guaranteed to be calibrated. A high score can still be wrong.

A simple policy accepts predictions at or above a threshold and sends the rest for review:

- **coverage:** fraction of all cases accepted automatically;
- **selective accuracy:** accuracy among accepted cases only.

Thresholds must be selected with validation data and business costs. The test results below are a diagnostic comparison, not a threshold-selection procedure.

In [ ]:
# WHAT: Compare automation coverage and selective accuracy across confidence thresholds.
# WHY: Product safety requires an explicit trade-off between automation and review.
# OUTPUT: A compact threshold-policy table on the held-out test predictions.

thresholds = [0.50, 0.70, 0.80, 0.90, 0.95]
print(f"{'threshold':>10} {'coverage':>10} {'selective_acc':>15} {'reviewed':>10}")

for threshold in thresholds:
    accepted = test_confidences >= threshold
    coverage = accepted.mean()
    selective_accuracy = (
        (test_predictions[accepted] == y_test_eval[accepted]).mean()
        if accepted.any()
        else float("nan")
    )
    reviewed = (~accepted).sum()
    print(
        f"{threshold:10.2f} {coverage:10.3f} "
        f"{selective_accuracy:15.3f} {reviewed:10d}"
    )

In [ ]:
# WHAT: Visualize confidence for correct and incorrect predictions.
# WHY: Overlapping distributions show why confidence is useful but imperfect.
# OUTPUT: Two confidence histograms using the same bins.

correct_mask = test_predictions == y_test_eval
bins = np.linspace(0, 1, 16)

plt.figure(figsize=(8, 4))
plt.hist(test_confidences[correct_mask], bins=bins, alpha=0.7, label="correct")
plt.hist(test_confidences[~correct_mask], bins=bins, alpha=0.7, label="incorrect")
plt.xlabel("Maximum softmax confidence")
plt.ylabel("Number of predictions")
plt.title("Confidence is informative, not a guarantee")
plt.legend()
plt.tight_layout()
plt.show()

## 13. Create a single-image prediction function

An inference boundary should validate its input, reproduce training preprocessing, add channel and batch dimensions, use evaluation mode, and return both the proposed class and decision information.

In [ ]:
# WHAT: Wrap preprocessing, inference, top-k probabilities, and the review decision.
# WHY: A clear prediction boundary reduces train-serving preprocessing mismatches.
# OUTPUT: One structured prediction for a raw 8x8 image.

def predict_digit(model, raw_image, threshold=0.80, device=device):
    raw_image = np.asarray(raw_image)
    if raw_image.shape != (8, 8):
        raise ValueError(f"Expected shape (8, 8), received {raw_image.shape}")
    if raw_image.min() < 0 or raw_image.max() > 16:
        raise ValueError("Expected raw pixel values in the range [0, 16]")

    image_tensor = (
        torch.from_numpy(raw_image.astype(np.float32) / 16.0)
        .unsqueeze(0)
        .unsqueeze(0)
        .to(device)
    )

    model.eval()
    with torch.inference_mode():
        logits = model(image_tensor)
        probabilities = torch.softmax(logits, dim=1)[0]

    confidence, predicted_index = probabilities.max(dim=0)
    top_values, top_indices = probabilities.topk(3)
    confidence_value = float(confidence.cpu())

    return {
        "prediction": class_names[int(predicted_index.cpu())],
        "confidence": confidence_value,
        "decision": "accept" if confidence_value >= threshold else "review",
        "top_3": [
            (class_names[int(index)], float(value))
            for value, index in zip(top_values.cpu(), top_indices.cpu())
        ],
    }

sample_result = predict_digit(model, X_test_raw[0], threshold=0.80)
print("True label:", y_test[0])
print("Prediction result:", sample_result)

## 14. Save a reproducible checkpoint and verify reload

Weights are not a complete inference artifact. The checkpoint also stores architecture settings, class names, input shape, pixel scaling, and the best epoch. A production registry should additionally track code revision, dataset version, package environment, evaluation report, and approval status.

In [ ]:
# WHAT: Save model state with its inference contract and experiment metadata.
# WHY: Preprocessing and label metadata are required to reproduce predictions safely.
# OUTPUT: A versioned checkpoint file under artifacts/.

artifact_dir = Path("artifacts")
artifact_dir.mkdir(exist_ok=True)
checkpoint_path = artifact_dir / "digits_cnn_checkpoint.pt"

checkpoint = {
    "format_version": 1,
    "model_name": "SmallCNN",
    "model_config": {"num_classes": 10, "dropout": 0.20},
    "state_dict": {key: value.cpu() for key, value in best_state.items()},
    "class_names": class_names.tolist(),
    "input_shape_chw": [1, 8, 8],
    "raw_pixel_range": [0, 16],
    "preprocessing": "float32(raw_image) / 16.0",
    "best_epoch": best_epoch,
    "best_validation_loss": best_valid_loss,
    "seed": SEED,
}

torch.save(checkpoint, checkpoint_path)
print(f"Saved: {checkpoint_path.resolve()}")
print(f"Size: {checkpoint_path.stat().st_size / 1024:.1f} KiB")

In [ ]:
# WHAT: Reload into a new model and compare logits with the current model.
# WHY: A successful save call does not prove that the artifact reproduces inference.
# OUTPUT: A strict numerical consistency check for the reloaded checkpoint.

try:
    loaded_checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
except TypeError:  # Compatibility with older PyTorch versions.
    loaded_checkpoint = torch.load(checkpoint_path, map_location="cpu")

reloaded_model = SmallCNN(**loaded_checkpoint["model_config"])
reloaded_model.load_state_dict(loaded_checkpoint["state_dict"])
reloaded_model.eval()

verification_batch = X_test[:8]
model_cpu = deepcopy(model).to("cpu").eval()

with torch.inference_mode():
    original_logits = model_cpu(verification_batch)
    reloaded_logits = reloaded_model(verification_batch)

torch.testing.assert_close(original_logits, reloaded_logits)
print("Reload verification passed: logits match.")
print("Stored preprocessing:", loaded_checkpoint["preprocessing"])
print("Stored classes:", loaded_checkpoint["class_names"])

## 15. Measure a local latency baseline

Latency depends on hardware, batch size, warm-up, preprocessing, and framework settings. This cell measures only the reloaded model's forward pass for one already-preprocessed CPU image. It is a local baseline, not a production SLA.

A deployment benchmark should measure the complete request path and report percentiles such as p50 and p95 on target hardware.

In [ ]:
# WHAT: Warm up and measure repeated single-image CPU forward passes.
# WHY: Model quality must be considered together with operational constraints.
# OUTPUT: Median and 95th-percentile local forward-pass latency.

latency_sample = X_test[:1]
reloaded_model.eval()

with torch.inference_mode():
    for _ in range(20):
        _ = reloaded_model(latency_sample)

    latency_ms = []
    for _ in range(200):
        start = time.perf_counter()
        _ = reloaded_model(latency_sample)
        latency_ms.append((time.perf_counter() - start) * 1000)

print(f"Local CPU p50 forward latency: {np.percentile(latency_ms, 50):.3f} ms")
print(f"Local CPU p95 forward latency: {np.percentile(latency_ms, 95):.3f} ms")

## 16. Practical exercises

### Exercise A — Output shape

Implement the formula for a one-dimensional convolution output. Check that an input of `32`, kernel `5`, padding `2`, and stride `2` produces `16`.

In [ ]:
# WHAT: Complete a convolution-output calculation.
# WHY: Shape arithmetic is a frequent source of CNN implementation errors.
# OUTPUT: Add your implementation, then print the result.

def exercise_output_size(size, kernel_size, padding, stride):
    # Replace pass with the integer output-size formula.
    pass

# result = exercise_output_size(32, kernel_size=5, padding=2, stride=2)
# print(result)
# assert result == 16

**Answer scaffold:** identify the numerator first, apply floor division by stride, and then add one. Keep the formula in a function so you can test multiple architecture choices.

In [ ]:
# WHAT: Provide a runnable reference implementation for Exercise A.
# WHY: Comparing an attempted solution with a tested function closes the feedback loop.
# OUTPUT: The expected output size, 16.

def exercise_output_size_solution(size, kernel_size, padding, stride):
    return (size + 2 * padding - kernel_size) // stride + 1

solution_result = exercise_output_size_solution(32, 5, 2, 2)
print(solution_result)
assert solution_result == 16

### Exercise B — Make a model comparison

Compare one controlled change, such as dropout `0.0` versus `0.2`, while keeping the split, seed, optimizer, learning rate, and maximum epochs fixed. Report:

- best validation loss and epoch;
- validation accuracy;
- test accuracy only for the final selected setting;
- parameter count and local latency;
- the error pattern that changed.

Do not choose a model from training accuracy alone.

**Answer scaffold:** create a small results dictionary with one row per configuration. Select the configuration by validation evidence, restore it, and then evaluate it once on test data.

### Exercise C — Design the review policy

Assume a wrong automatic route costs 20 units and a manual review costs 1 unit. On the validation set, compare thresholds and calculate:

```text
total cost = 20 × accepted mistakes + 1 × reviewed cases
```

Choose the threshold with the lowest validation cost, then report its test coverage and selective accuracy.

**Answer scaffold:** get validation logits from `evaluate`, apply softmax, loop over candidate thresholds, and store accepted mistakes, reviewed cases, and total cost. Keep the test set out of threshold selection.